In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from pathlib import Path
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix

from xgboost import XGBClassifier, plot_importance
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report,
    confusion_matrix
)
from sklearn.ensemble import RandomForestClassifier


pd.set_option("display.max_columns", 120)


In [15]:
X = pd.read_csv("Client.csv")
y = pd.read_csv("Record.csv")

df = pd.merge(X, y, on="Customer_ID", how="inner")

print(df["churn"].value_counts())
print(df["churn"].value_counts(normalize=True) * 100)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

# Inspect the target
print("\nPossible target columns:")
print([
    col for col in df.columns
    if "churn" in col.lower() or "stay" in col.lower()
])

# Inspect the first rows
# display(df.head())

# Check missing values
print("\nMissing values:")
print(df.isnull().sum().sort_values(ascending=False).head(15))

churn
0    50438
1    49562
Name: count, dtype: int64
churn
0    50.438
1    49.562
Name: proportion, dtype: float64
Dataset shape: (100000, 100)

Columns:
['uniqsubs', 'actvsubs', 'new_cell', 'crclscod', 'asl_flag', 'totcalls', 'totmou', 'totrev', 'adjrev', 'adjmou', 'adjqty', 'avgrev', 'avgmou', 'avgqty', 'avg3mou', 'avg3qty', 'avg3rev', 'avg6mou', 'avg6qty', 'avg6rev', 'prizm_social_one', 'area', 'dualband', 'refurb_new', 'hnd_price', 'phones', 'models', 'hnd_webcap', 'truck', 'rv', 'ownrent', 'lor', 'dwlltype', 'marital', 'adults', 'infobase', 'income', 'numbcars', 'HHstatin', 'dwllsize', 'forgntvl', 'ethnic', 'kid0_2', 'kid3_5', 'kid6_10', 'kid11_15', 'kid16_17', 'creditcd', 'eqpdays', 'Customer_ID', 'rev_Mean', 'mou_Mean', 'totmrc_Mean', 'da_Mean', 'ovrmou_Mean', 'ovrrev_Mean', 'vceovr_Mean', 'datovr_Mean', 'roam_Mean', 'change_mou', 'change_rev', 'drop_vce_Mean', 'drop_dat_Mean', 'blck_vce_Mean', 'blck_dat_Mean', 'unan_vce_Mean', 'unan_dat_Mean', 'plcd_vce_Mean', 'plcd_dat_Mean'

In [20]:

# 1. Verify target
assert "churn" in df.columns, (
    "The churn target is missing from df. "
    "Check how the target was loaded or joined."
)

print("Shape:", df.shape)
print(df["churn"].value_counts(dropna=False))

# 2. Remove rows without a target
df = df.dropna(subset=["churn"]).copy()

# Customer_ID is an identifier, not a predictive feature
X = df.drop(columns=["Customer_ID", "churn"], errors="ignore")
y = df["churn"]

# 3. Identify categorical and numerical columns
categorical_cols = X.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

numerical_cols = X.select_dtypes(
    include=["number"]
).columns.tolist()

# Convert numeric columns containing inf to missing values
X[numerical_cols] = X[numerical_cols].replace(
    [np.inf, -np.inf], np.nan
)




# 4. Preprocessing
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numerical_cols),
    ("cat", categorical_pipeline, categorical_cols)
])

# 5. Stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

# 6. Compare models
models = {

    # "Random Forest": RandomForestClassifier(
    #     n_estimators=300,
    #     min_samples_leaf=2,
    #     n_jobs=-1,
    #     random_state=42
    # ),
    # "XGBoost": XGBLinear(
    #     max_iter=200,
    #     learning_rate=0.08,
    #     max_leaf_nodes=31,
    #     l2_regularization=1.0,
    #     random_state=42
    # ),
     "XGBoost (Tree)": XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=2,
    tree_method="hist",
    n_jobs=-1,
    random_state=42
)

}

results = []

for name, model in models.items():
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)

    predictions = pipeline.predict(X_test)
    probabilities = pipeline.predict_proba(X_test)[:, 1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(
            y_test, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_test, predictions, zero_division=0
        ),
        "F1": f1_score(
            y_test, predictions, zero_division=0
        ),
        "ROC-AUC": roc_auc_score(y_test, probabilities)
    })

results_df = pd.DataFrame(results).sort_values(
    "ROC-AUC", ascending=False
)

display(results_df.round(4))

Shape: (100000, 100)
churn
0    50438
1    49562
Name: count, dtype: int64


/tmp/ipykernel_29814/1990034070.py:18: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(


,Model,Accuracy,Precision,Recall,F1,ROC-AUC
0,XGBoost (Tree),0.6402,0.6344,0.6467,0.6405,0.6953



# Customer Churn Dataset — Column Descriptions

## 1. Account Activity and Usage

| Column | Description | Expected Type |
|---|---|---|
| `uniqsubs` | Number of unique subscriptions | Numerical |
| `actvsubs` | Number of active subscriptions | Numerical |
| `new_cell` | New cellular customer or service indicator | Categorical |
| `crclscod` | Customer classification or rate-plan code | Categorical |
| `asl_flag` | Special service or account indicator | Categorical |
| `totcalls` | Total number of calls | Numerical |
| `totmou` | Total minutes of use | Numerical |
| `totrev` | Total revenue | Numerical |
| `adjrev` | Adjusted revenue | Numerical |
| `adjmou` | Adjusted minutes of use | Numerical |
| `adjqty` | Adjusted usage quantity | Numerical |
| `avgrev` | Average revenue | Numerical |
| `avgmou` | Average minutes of use | Numerical |
| `avgqty` | Average usage quantity | Numerical |
| `avg3mou` | Average minutes of use over three months | Numerical |
| `avg3qty` | Average usage quantity over three months | Numerical |
| `avg3rev` | Average revenue over three months | Numerical |
| `avg6mou` | Average minutes of use over six months | Numerical |
| `avg6qty` | Average usage quantity over six months | Numerical |
| `avg6rev` | Average revenue over six months | Numerical |

## 2. Phone and Device Information

| Column | Description | Expected Type |
|---|---|---|
| `dualband` | Phone's dual-band capability or category | Categorical |
| `refurb_new` | Whether the phone is refurbished or new | Categorical |
| `hnd_price` | Recorded handset price | Numerical |
| `phones` | Number of phones associated with the customer | Numerical |
| `models` | Number of phone models associated with the customer | Numerical |
| `hnd_webcap` | Handset web capability or category | Categorical |
| `eqpdays` | Number of days associated with handset age | Numerical |

## 3. Geographic, Housing, and Lifestyle Information

| Column | Description | Expected Type |
|---|---|---|
| `prizm_social_one` | PRIZM social or marketing segment | Categorical |
| `area` | Geographic area or area code | Categorical |
| `truck` | Truck ownership or related indicator | Categorical |
| `rv` | Recreational vehicle ownership or related indicator | Categorical |
| `ownrent` | Whether the residence is owned or rented | Categorical |
| `lor` | Length of residence | Numerical or categorical |
| `dwlltype` | Dwelling type | Categorical |
| `dwllsize` | Dwelling size or size category | Categorical |
| `forgntvl` | Foreign travel indicator | Categorical |
| `ethnic` | Recorded ethnic category | Categorical |

## 4. Household and Demographic Information

| Column | Description | Expected Type |
|---|---|---|
| `marital` | Marital status | Categorical |
| `adults` | Number of adults in the household | Numerical |
| `infobase` | Information or customer-record indicator | Categorical |
| `income` | Income amount or income category | Numerical or categorical |
| `numbcars` | Number of cars in the household | Numerical |
| `HHstatin` | Household status indicator or category | Categorical |
| `kid0_2` | Indicator or category for children aged 0–2 | Categorical |
| `kid3_5` | Indicator or category for children aged 3–5 | Categorical |
| `kid6_10` | Indicator or category for children aged 6–10 | Categorical |
| `kid11_15` | Indicator or category for children aged 11–15 | Categorical |
| `kid16_17` | Indicator or category for children aged 16–17 | Categorical |

## 5. Credit Information and Target

| Column | Description | Expected Type |
|---|---|---|
| `creditcd` | Credit card or credit-related indicator | Categorical |
| `Customer_ID` | Unique customer identifier | Identifier |
| `churn` | Target indicating whether the customer left the company | Binary (0/1) |

## 6. Machine Learning Considerations

| Category | Recommended Action |
|---|---|
| Numerical features | Check missing values, outliers, and distributions |
| Categorical features | Inspect unique values and encode appropriately |
| `Customer_ID` | Exclude from model features by default |
| `churn` | Use as the target variable `y`, never as an input feature |
| Sensitive features | Review privacy, fairness, and permitted use before training |

**Note:** Column meanings and types are inferred from their names. Verify them against the dataset's documentation and actual values before preprocessing.



# Feature Priority for Customer Churn Prediction

Features are grouped by their expected usefulness before measuring actual feature importance.

## 1. High-Priority Features

These describe customer usage, spending, and device age, which may help predict churn.

| Column | Description | Reason for Priority |
|---|---|---|
| `avg3rev` | Average revenue over three months | Recent spending behavior |
| `avg3mou` | Average minutes of use over three months | Recent usage behavior |
| `avg3qty` | Average usage quantity over three months | Recent usage activity |
| `avg6rev` | Average revenue over six months | Medium-term spending patterns |
| `avg6mou` | Average minutes of use over six months | Medium-term usage patterns |
| `avgrev` | Average revenue | General customer spending |
| `avgmou` | Average minutes of use | General customer activity |
| `eqpdays` | Days associated with handset age | Device replacement or upgrade patterns |
| `totrev` | Total revenue | Overall customer value |
| `totmou` | Total minutes of use | Overall usage level |
| `totcalls` | Total number of calls | Calling activity |
| `adjrev` | Adjusted revenue | Alternative measure of customer spending |

## 2. Medium-Priority Features

These describe subscription status, handset characteristics, and customer account details.

| Column | Description | Reason for Priority |
|---|---|---|
| `actvsubs` | Number of active subscriptions | Account activity |
| `uniqsubs` | Number of unique subscriptions | Subscription behavior |
| `adjmou` | Adjusted minutes of use | Adjusted usage patterns |
| `adjqty` | Adjusted usage quantity | Adjusted activity |
| `hnd_price` | Recorded handset price | Device value |
| `phones` | Number of associated phones | Device ownership patterns |
| `models` | Number of associated phone models | Device diversity |
| `hnd_webcap` | Handset web capability | Technology access |
| `dualband` | Dual-band phone capability | Device characteristics |
| `refurb_new` | Refurbished or new handset category | Device condition |
| `crclscod` | Customer classification or rate-plan code | Service or customer segment |
| `new_cell` | New cellular customer or service indicator | Customer status |
| `asl_flag` | Special account or service indicator | Account characteristics |
| `creditcd` | Credit-related indicator | Customer account characteristics |

## 3. Lower-Priority Features to Test

These demographic, household, geographic, and lifestyle features may provide additional predictive information, but their usefulness must be demonstrated empirically.

| Column | Description |
|---|---|
| `area` | Geographic area |
| `prizm_social_one` | Social or marketing segment |
| `ownrent` | Home ownership or rental status |
| `lor` | Length of residence |
| `dwlltype` | Dwelling type |
| `dwllsize` | Dwelling size category |
| `marital` | Marital status |
| `adults` | Number of adults in the household |
| `income` | Income amount or category |
| `numbcars` | Number of cars |
| `HHstatin` | Household status category |
| `infobase` | Information or customer-record indicator |
| `kid0_2` | Children aged 0–2 indicator or category |
| `kid3_5` | Children aged 3–5 indicator or category |
| `kid6_10` | Children aged 6–10 indicator or category |
| `kid11_15` | Children aged 11–15 indicator or category |
| `kid16_17` | Children aged 16–17 indicator or category |
| `truck` | Truck ownership or related indicator |
| `rv` | Recreational vehicle ownership or related indicator |
| `forgntvl` | Foreign travel indicator |
| `ethnic` | Recorded ethnic category |

## 4. Exclude from the Input Features by Default

| Column | Reason |
|---|---|
| `Customer_ID` | Identifier; may encourage memorization rather than generalization |
| `churn` | Target variable; including it in `X` causes target leakage |

## Important Notes

- This is an initial investigation order, not a verified feature-importance ranking.
- Some features are strongly correlated, such as `avg3rev`, `avg6rev`, `avgrev`, and `totrev`.
- Review the time window and definition of every feature to ensure it was available before the churn prediction date.
- Evaluate sensitive features such as `ethnic` carefully for privacy, fairness, and permitted use.
- Use validation-based feature selection rather than removing columns solely because they appear less important.
